# 01 — Data Setup & Infrastructure

**Purpose**: Foundation notebook for the wine-diversification correlation analysis.  
All downstream notebooks (02–06) depend on the five parquet files saved here.

## Sections
1. Environment setup
2. MotherDuck connection & schema discovery
3. Liv-ex index history (CSV) → `livex_indices_clean.parquet`
4. Comparison assets from yfinance → `comparison_assets_monthly.parquet`
5. Focal wines VWAP (Salon, Dom Pérignon, Lafite) → `focal_wines_vwap_monthly.parquet`
6. Top 30 most-traded LWIN7s VWAP → `top30_wines_vwap_monthly.parquet`
7. Monthly bid data → `bids_monthly.parquet`
8. Data quality assertions

**All prices in GBP unless otherwise stated.**

## 1. Environment Setup

In [ ]:
import os
import warnings
import duckdb
import pandas as pd
import yfinance as yf
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paths — notebook lives in projects/correlation-diversification/notebooks/
# When Jupyter runs, CWD is the notebook directory.
# DATA_DIR is one level up: projects/correlation-diversification/data/
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path("__file__").resolve().parent   # .../notebooks/
PROJECT_DIR  = NOTEBOOK_DIR.parent                  # .../correlation-diversification/
DATA_DIR     = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_CSV              = DATA_DIR / "liv-ex_index_history.csv"
LIVEX_PARQUET          = DATA_DIR / "livex_indices_clean.parquet"
COMPARISON_PARQUET     = DATA_DIR / "comparison_assets_monthly.parquet"
FOCAL_WINES_PARQUET    = DATA_DIR / "focal_wines_vwap_monthly.parquet"
TOP30_WINES_PARQUET    = DATA_DIR / "top30_wines_vwap_monthly.parquet"
BIDS_PARQUET           = DATA_DIR / "bids_monthly.parquet"

print("Notebook dir:", NOTEBOOK_DIR)
print("Data dir:    ", DATA_DIR)
print("motherduck_token present:", bool(os.getenv("motherduck_token")))
print("Liv-ex CSV exists:", LIVEX_CSV.exists())

## 2. MotherDuck Connection & Schema Discovery

Connect to MotherDuck (`duckdb.connect("md:")`) and query `information_schema.columns`
for both tables **before** touching any row data. We never assume column names.

**Tables**
- `winefi.ml.ml_unified_trades_tbvm` — unified trade data (tens of millions of rows)
- `winefi.mrt.mrt_unified_bids` — unified bid data

**Performance rules**: never `SELECT *` without `LIMIT`; push aggregation into SQL.

In [ ]:
con = duckdb.connect("md:")
print("Connected to MotherDuck")

In [ ]:
# Schema: ml_unified_trades_tbvm
trades_schema = con.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_catalog = 'winefi'
      AND table_schema  = 'ml'
      AND table_name    = 'ml_unified_trades_tbvm'
    ORDER BY ordinal_position
""").df()

print("=== winefi.ml.ml_unified_trades_tbvm ===")
print(f"Columns discovered: {len(trades_schema)}")
display(trades_schema)

In [ ]:
# Schema: mrt_unified_bids
bids_schema = con.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_catalog = 'winefi'
      AND table_schema  = 'mrt'
      AND table_name    = 'mrt_unified_bids'
    ORDER BY ordinal_position
""").df()

print("=== winefi.mrt.mrt_unified_bids ===")
print(f"Columns discovered: {len(bids_schema)}")
display(bids_schema)

In [ ]:
# ---------------------------------------------------------------------------
# Dynamic column detection helpers
# ---------------------------------------------------------------------------

def first_matching_col(schema_df, *patterns):
    """Return first actual column name whose lowercase form contains any pattern."""
    for _, row in schema_df.iterrows():
        col_lower = row["column_name"].lower()
        for p in patterns:
            if p.lower() in col_lower:
                return row["column_name"]
    return None

# --- Trades table ---
TRADES_DATE_COL   = first_matching_col(trades_schema, "trade_date", "transaction_date", "date", "time")
TRADES_LWIN7_COL  = first_matching_col(trades_schema, "lwin7", "lwin_7")
TRADES_LWIN11_COL = first_matching_col(trades_schema, "lwin11", "lwin_11")
TRADES_LWIN16_COL = first_matching_col(trades_schema, "lwin16", "lwin_16")
TRADES_LWIN18_COL = first_matching_col(trades_schema, "lwin18", "lwin_18")
TRADES_PRICE_COL  = first_matching_col(trades_schema, "price_gbp", "trade_price", "price", "value")
TRADES_QTY_COL    = first_matching_col(trades_schema, "quantity", "qty", "volume", "bottles", "amount")
TRADES_BOTTLE_COL = first_matching_col(trades_schema, "bottle_size", "format", "bottle")

print("=== Detected columns for ml_unified_trades_tbvm ===")
print(f"  Date column:        {TRADES_DATE_COL}")
print(f"  LWIN7 column:       {TRADES_LWIN7_COL}")
print(f"  LWIN11 column:      {TRADES_LWIN11_COL}")
print(f"  LWIN16 column:      {TRADES_LWIN16_COL}")
print(f"  LWIN18 column:      {TRADES_LWIN18_COL}")
print(f"  Price column:       {TRADES_PRICE_COL}")
print(f"  Quantity column:    {TRADES_QTY_COL}")
print(f"  Bottle size column: {TRADES_BOTTLE_COL}")

# Validate required columns
if TRADES_DATE_COL is None:
    raise ValueError(f"No date column found in trades table. Columns: {trades_schema['column_name'].tolist()}")
if TRADES_PRICE_COL is None:
    raise ValueError(f"No price column found in trades table. Columns: {trades_schema['column_name'].tolist()}")

# Build LWIN7 SQL expression (prefer direct lwin7 col, else extract from longer key)
if TRADES_LWIN7_COL:
    LWIN7_EXPR = f'CAST("{TRADES_LWIN7_COL}" AS VARCHAR)'
elif TRADES_LWIN11_COL:
    LWIN7_EXPR = f'LEFT(CAST("{TRADES_LWIN11_COL}" AS VARCHAR), 7)'
elif TRADES_LWIN16_COL:
    LWIN7_EXPR = f'LEFT(CAST("{TRADES_LWIN16_COL}" AS VARCHAR), 7)'
elif TRADES_LWIN18_COL:
    LWIN7_EXPR = f'LEFT(CAST("{TRADES_LWIN18_COL}" AS VARCHAR), 7)'
else:
    raise ValueError(f"No LWIN identifier found in trades table. Columns: {trades_schema['column_name'].tolist()}")

print(f"\n  LWIN7 SQL expression: {LWIN7_EXPR}")

# Build 750ml bottle-size filter
if TRADES_BOTTLE_COL:
    BOTTLE_FILTER = f'AND CAST("{TRADES_BOTTLE_COL}" AS DOUBLE) = 750'
elif TRADES_LWIN16_COL:
    # LWIN16: LWIN7(7) + vintage(4) + bottle_size_zero_padded(5) = 16 chars
    # chars 12-16 encode bottle size, e.g. '00750' = 750ml
    BOTTLE_FILTER = (
        f'AND CAST(SUBSTRING(LPAD(CAST("{TRADES_LWIN16_COL}" AS VARCHAR), 16, \'0\'), 12, 5) AS INTEGER) = 750'
    )
elif TRADES_LWIN18_COL:
    # LWIN18: LWIN7(7) + vintage(4) + pack_size(2) + bottle_size(5) = 18 chars
    BOTTLE_FILTER = (
        f'AND CAST(SUBSTRING(LPAD(CAST("{TRADES_LWIN18_COL}" AS VARCHAR), 18, \'0\'), 14, 5) AS INTEGER) = 750'
    )
else:
    print("WARNING: No bottle size column detected — 750ml filter will NOT be applied.")
    BOTTLE_FILTER = ""

print(f"  Bottle filter: {BOTTLE_FILTER or '(none)'}")

In [ ]:
# --- Bids table ---
BIDS_DATE_COL  = first_matching_col(bids_schema, "bid_date", "date", "time", "created", "updated")
BIDS_PRICE_COL = first_matching_col(bids_schema, "price_gbp", "bid_price", "price", "value", "amount")

print("=== Detected columns for mrt_unified_bids ===")
print(f"  Date column:  {BIDS_DATE_COL}")
print(f"  Price column: {BIDS_PRICE_COL}")

if BIDS_DATE_COL is None:
    raise ValueError(f"No date column found in bids table. Columns: {bids_schema['column_name'].tolist()}")
if BIDS_PRICE_COL is None:
    raise ValueError(f"No price column found in bids table. Columns: {bids_schema['column_name'].tolist()}")

In [ ]:
# Lightweight row-count + date-range preview (all aggregation in SQL — no SELECT *)
trades_summary = con.execute(f"""
    SELECT
        COUNT(*)                                       AS row_count,
        MIN(CAST(\"{TRADES_DATE_COL}\" AS DATE))      AS min_date,
        MAX(CAST(\"{TRADES_DATE_COL}\" AS DATE))      AS max_date
    FROM winefi.ml.ml_unified_trades_tbvm
    WHERE CAST(\"{TRADES_DATE_COL}\" AS DATE) >= '2000-01-01'
""").df()

bids_summary = con.execute(f"""
    SELECT
        COUNT(*)                                     AS row_count,
        MIN(CAST(\"{BIDS_DATE_COL}\" AS DATE))      AS min_date,
        MAX(CAST(\"{BIDS_DATE_COL}\" AS DATE))      AS max_date
    FROM winefi.mrt.mrt_unified_bids
    WHERE CAST(\"{BIDS_DATE_COL}\" AS DATE) >= '2000-01-01'
""").df()

print("Trades table summary (from 2000):")
display(trades_summary)
print("Bids table summary (from 2000):")
display(bids_summary)

## 3. Liv-ex Index History (CSV)

Load `data/liv-ex_index_history.csv`, inspect its structure, clean to 2000+, resample to  
month-end frequency, and save as `livex_indices_clean.parquet`.  
All numeric index columns in the file are preserved.

In [ ]:
# Try reading with a 'date' column as index; fall back to first column
try:
    livex_raw = pd.read_csv(LIVEX_CSV, index_col="date", parse_dates=True)
    if livex_raw.index.dtype == "object":
        raise ValueError("date column not parsed as datetime")
except (ValueError, KeyError):
    livex_raw = pd.read_csv(LIVEX_CSV, index_col=0, parse_dates=True)

print("=== Liv-ex CSV — raw structure ===")
print(f"Shape:      {livex_raw.shape}")
print(f"Index type: {livex_raw.index.dtype}")
print(f"Date range: {livex_raw.index.min()} → {livex_raw.index.max()}")
print()
print("All columns and dtypes:")
print(livex_raw.dtypes)
print()
livex_raw.head()

In [ ]:
# Retain only numeric index columns; drop metadata (created_at, updated_at, load_ts, etc.)
numeric_cols = livex_raw.select_dtypes(include="number").columns.tolist()
livex_indices = livex_raw[numeric_cols].copy()
livex_indices.index = pd.to_datetime(livex_indices.index)

# Filter to 2000 onwards and resample to calendar month-end
livex_indices = livex_indices[livex_indices.index >= "2000-01-01"]
livex_monthly = livex_indices.resample("ME").last()

print("=== Liv-ex — monthly aligned ===")
print(f"Shape:      {livex_monthly.shape}")
print(f"Date range: {livex_monthly.index.min().date()} → {livex_monthly.index.max().date()}")
print()

col_doc = pd.DataFrame({
    "column":   livex_monthly.columns,
    "dtype":    livex_monthly.dtypes.values,
    "non_null": livex_monthly.notna().sum().values,
    "min":      livex_monthly.min().values,
    "max":      livex_monthly.max().values,
})
print("Column documentation:")
display(col_doc)

livex_monthly.to_parquet(LIVEX_PARQUET)
print(f"\nSaved → {LIVEX_PARQUET}")
print(f"File size: {LIVEX_PARQUET.stat().st_size / 1024:.1f} KB")

## 4. Comparison Assets from yfinance

Pull monthly `Close` prices for:
- **S&P 500** (`^GSPC`) — US equity benchmark
- **FTSE 100** (`^FTSE`) — UK equity benchmark (same GBP base as Liv-ex)
- **Gold** (`GC=F`) — safe-haven comparison asset

All from 2000-01-01, resampled to month-end. Saved alongside Liv-ex in one combined parquet.

In [ ]:
TICKERS = {
    "sp500":   "^GSPC",
    "ftse100": "^FTSE",
    "gold":    "GC=F",
}
START_DATE = "2000-01-01"

frames = {}
for name, ticker in TICKERS.items():
    raw = yf.download(ticker, start=START_DATE, progress=False, auto_adjust=False)["Close"]
    if isinstance(raw, pd.DataFrame):
        raw = raw.squeeze()
    frames[name] = raw
    print(
        f"{name} ({ticker}): {len(raw)} daily rows, "
        f"{raw.index.min().date()} → {raw.index.max().date()}, "
        f"nulls={raw.isna().sum()}"
    )

comparison_daily = pd.DataFrame(frames)
print(f"\nCombined daily shape: {comparison_daily.shape}")
comparison_daily.head()

In [ ]:
# Resample to month-end and combine with Liv-ex indices
comparison_monthly = comparison_daily.resample("ME").last()
combined = comparison_monthly.join(livex_monthly, how="outer")
combined = combined[combined.index >= "2000-01-01"]

print("=== Combined monthly dataset ===")
print(f"Shape:      {combined.shape}")
print(f"Date range: {combined.index.min().date()} → {combined.index.max().date()}")
print()
print("Columns:")
for col in combined.columns:
    non_null = combined[col].notna().sum()
    print(f"  {col:<42} dtype={combined[col].dtype}  non_null={non_null}")

combined.to_parquet(COMPARISON_PARQUET)
print(f"\nSaved → {COMPARISON_PARQUET}")
print(f"File size: {COMPARISON_PARQUET.stat().st_size / 1024:.1f} KB")

## 5. Focal Wines Monthly VWAP

Pull aggregated monthly VWAP trade data for three focal wines:
- **Salon** (LWIN7 = 1807626) — highly resilient in market downturns
- **Dom Pérignon** (LWIN7 = 1082656) — benchmark Champagne
- **Lafite** (LWIN7 = 1011872) — most liquid Bordeaux

Filter: 750ml bottles only. All aggregation is done in SQL.

In [ ]:
FOCAL_LWIN7S = ["1807626", "1082656", "1011872"]
FOCAL_LABELS = {"1807626": "Salon", "1082656": "Dom Perignon", "1011872": "Lafite"}

focal_in_list = ", ".join(f"'{lw}'" for lw in FOCAL_LWIN7S)

# VWAP = weighted average price: SUM(price*qty) / SUM(qty)
# Fall back to simple average if no quantity column available
if TRADES_QTY_COL:
    price_x_qty = f'CAST("{TRADES_PRICE_COL}" AS DOUBLE) * CAST("{TRADES_QTY_COL}" AS DOUBLE)'
    vwap_expr   = f'SUM({price_x_qty}) / NULLIF(SUM(CAST("{TRADES_QTY_COL}" AS DOUBLE)), 0)'
    qty_expr    = f'SUM(CAST("{TRADES_QTY_COL}" AS DOUBLE)) AS total_qty'
else:
    print("WARNING: No quantity column — using AVG(price) as VWAP proxy.")
    vwap_expr = f'AVG(CAST("{TRADES_PRICE_COL}" AS DOUBLE))'
    qty_expr  = 'NULL AS total_qty'

focal_query = f"""
    SELECT
        DATE_TRUNC('month', CAST("{TRADES_DATE_COL}" AS DATE))  AS month,
        {LWIN7_EXPR}                                             AS lwin7,
        {vwap_expr}                                              AS vwap_gbp,
        {qty_expr},
        COUNT(*)                                                 AS trade_count
    FROM winefi.ml.ml_unified_trades_tbvm
    WHERE
        {LWIN7_EXPR} IN ({focal_in_list})
        AND CAST("{TRADES_DATE_COL}" AS DATE) >= '2000-01-01'
        {BOTTLE_FILTER}
    GROUP BY 1, 2
    ORDER BY 1, 2
"""

print("Focal wines SQL query:")
print(focal_query)

In [ ]:
focal_df = con.execute(focal_query).df()
focal_df["month"]     = pd.to_datetime(focal_df["month"])
focal_df["wine_name"] = focal_df["lwin7"].map(FOCAL_LABELS)

print(f"Focal wines result: {focal_df.shape}")
print(f"Date range: {focal_df['month'].min().date()} → {focal_df['month'].max().date()}")
print()
print("Summary by wine:")
display(focal_df.groupby("wine_name")[["trade_count"]].agg(["sum", "count"]).rename(columns={"sum": "total_trades", "count": "months_with_data"}))
print()
focal_df.head(9)

In [ ]:
focal_df.to_parquet(FOCAL_WINES_PARQUET, index=False)
print(f"Saved → {FOCAL_WINES_PARQUET}")
print(f"File size: {FOCAL_WINES_PARQUET.stat().st_size / 1024:.1f} KB")

# Wide-format preview
focal_wide = focal_df.pivot_table(
    index="month", columns="wine_name", values="vwap_gbp", aggfunc="first"
)
print("\nWide-format preview — VWAP GBP per month (last 12 months):")
focal_wide.tail(12)

## 6. Top 30 Most-Traded LWIN7s — Monthly VWAP

Identify the top 30 LWIN7 wine labels by total trade count since 2000, then pull their  
monthly VWAP. This broader dataset supports custom index construction and heterogeneity  
analysis across wine labels.

Two-step approach: (1) get top 30 IDs via aggregation query, (2) pull VWAP for those IDs.

In [ ]:
# Step 1: Identify top 30 LWIN7s by trade count — pure aggregation, no row retrieval
top30_ids_query = f"""
    SELECT
        {LWIN7_EXPR}   AS lwin7,
        COUNT(*)        AS trade_count
    FROM winefi.ml.ml_unified_trades_tbvm
    WHERE CAST("{TRADES_DATE_COL}" AS DATE) >= '2000-01-01'
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 30
"""

top30_ids = con.execute(top30_ids_query).df()
print(f"Top 30 LWIN7s by trade count:")
display(top30_ids)

In [ ]:
# Step 2: Monthly VWAP for those top 30 LWIN7s
# Uses the IN list from Step 1 — no table alias complexity, no fragile string manipulation
top30_in_list = ", ".join(f"'{lw}'" for lw in top30_ids["lwin7"].tolist())

top30_query = f"""
    SELECT
        DATE_TRUNC('month', CAST("{TRADES_DATE_COL}" AS DATE))  AS month,
        {LWIN7_EXPR}                                             AS lwin7,
        {vwap_expr}                                              AS vwap_gbp,
        {qty_expr},
        COUNT(*)                                                 AS trade_count
    FROM winefi.ml.ml_unified_trades_tbvm
    WHERE
        CAST("{TRADES_DATE_COL}" AS DATE) >= '2000-01-01'
        AND {LWIN7_EXPR} IN ({top30_in_list})
    GROUP BY 1, 2
    ORDER BY 1, 2
"""

print("Top 30 VWAP query:")
print(top30_query)

In [ ]:
top30_df = con.execute(top30_query).df()
top30_df["month"] = pd.to_datetime(top30_df["month"])

print(f"Top 30 wines result: {top30_df.shape}")
print(f"Date range:    {top30_df['month'].min().date()} → {top30_df['month'].max().date()}")
print(f"Unique LWIN7s: {top30_df['lwin7'].nunique()}")
print()
coverage = top30_df.groupby("month")["lwin7"].count()
print(f"Monthly coverage (# LWIN7s with trades per month): mean={coverage.mean():.1f}, min={coverage.min()}, max={coverage.max()}")
top30_df.head()

In [ ]:
top30_df.to_parquet(TOP30_WINES_PARQUET, index=False)
print(f"Saved → {TOP30_WINES_PARQUET}")
print(f"File size: {TOP30_WINES_PARQUET.stat().st_size / 1024:.1f} KB")

## 7. Monthly Bid Data

Pull aggregated monthly bid price data from `winefi.mrt.mrt_unified_bids`.  
Aggregate to median bid price per month as a market-wide bid level indicator.  
Columns discovered dynamically in Section 2.

In [ ]:
bids_query = f"""
    SELECT
        DATE_TRUNC('month', CAST("{BIDS_DATE_COL}" AS DATE))    AS month,
        MEDIAN(CAST("{BIDS_PRICE_COL}" AS DOUBLE))              AS median_bid_gbp,
        AVG(CAST("{BIDS_PRICE_COL}" AS DOUBLE))                 AS mean_bid_gbp,
        MIN(CAST("{BIDS_PRICE_COL}" AS DOUBLE))                 AS min_bid_gbp,
        MAX(CAST("{BIDS_PRICE_COL}" AS DOUBLE))                 AS max_bid_gbp,
        COUNT(*)                                                 AS bid_count
    FROM winefi.mrt.mrt_unified_bids
    WHERE CAST("{BIDS_DATE_COL}" AS DATE) >= '2000-01-01'
    GROUP BY 1
    ORDER BY 1
"""

print("Bids query:")
print(bids_query)

In [ ]:
bids_df = con.execute(bids_query).df()
bids_df["month"] = pd.to_datetime(bids_df["month"])

print(f"Bids monthly: {bids_df.shape}")
print(f"Date range: {bids_df['month'].min().date()} → {bids_df['month'].max().date()}")
print()
display(bids_df.describe().round(2))

In [ ]:
bids_df.to_parquet(BIDS_PARQUET, index=False)
print(f"Saved → {BIDS_PARQUET}")
print(f"File size: {BIDS_PARQUET.stat().st_size / 1024:.1f} KB")
print()
bids_df.tail(6)

## 8. Data Quality Assertions

All five parquet files must pass these checks before the notebook is considered complete.

In [ ]:
errors = []

# 1. livex_indices_clean.parquet
livex_check = pd.read_parquet(LIVEX_PARQUET)
if len(livex_check) < 100:
    errors.append(f"livex_indices_clean has only {len(livex_check)} rows (need > 100)")
if livex_check.index.min().year > 2001:
    errors.append(f"livex_indices_clean starts in {livex_check.index.min().year} (need <= 2001)")
if livex_check.index.max().year < 2015:
    errors.append(f"livex_indices_clean ends in {livex_check.index.max().year} (need >= 2015)")

# 2. comparison_assets_monthly.parquet
comp_check = pd.read_parquet(COMPARISON_PARQUET)
if len(comp_check) < 100:
    errors.append(f"comparison_assets_monthly has only {len(comp_check)} rows (need > 100)")
if comp_check.index.min().year > 2001:
    errors.append(f"comparison_assets_monthly starts in {comp_check.index.min().year} (need <= 2001)")
for col in ["sp500", "ftse100", "gold"]:
    if col not in comp_check.columns:
        errors.append(f"comparison_assets_monthly missing column '{col}'")

# 3. focal_wines_vwap_monthly.parquet
focal_check = pd.read_parquet(FOCAL_WINES_PARQUET)
if len(focal_check) == 0:
    errors.append("focal_wines_vwap_monthly is empty")
for col in ["month", "lwin7", "vwap_gbp"]:
    if col not in focal_check.columns:
        errors.append(f"focal_wines_vwap_monthly missing required column '{col}'")
if "lwin7" in focal_check.columns:
    found   = set(focal_check["lwin7"].astype(str).unique())
    missing = set(["1807626", "1082656", "1011872"]) - found
    if missing:
        errors.append(f"focal_wines_vwap_monthly missing LWIN7s: {missing}")

# 4. top30_wines_vwap_monthly.parquet
top30_check = pd.read_parquet(TOP30_WINES_PARQUET)
if len(top30_check) == 0:
    errors.append("top30_wines_vwap_monthly is empty")
if "lwin7" in top30_check.columns and top30_check["lwin7"].nunique() < 10:
    errors.append(
        f"top30_wines_vwap_monthly has only {top30_check['lwin7'].nunique()} unique LWIN7s (expected ~30)"
    )

# 5. bids_monthly.parquet
bids_check = pd.read_parquet(BIDS_PARQUET)
if len(bids_check) == 0:
    errors.append("bids_monthly is empty")
if "median_bid_gbp" not in bids_check.columns:
    errors.append("bids_monthly missing column 'median_bid_gbp'")

# Report
if errors:
    for err in errors:
        print(f"FAIL: {err}")
    raise AssertionError("Data quality checks failed — see output above")
else:
    print("All data quality assertions PASSED.")
    print(f"  livex_indices_clean:        {len(livex_check)} rows, "
          f"{livex_check.index.min().date()} → {livex_check.index.max().date()}")
    print(f"  comparison_assets_monthly:  {len(comp_check)} rows, "
          f"{comp_check.index.min().date()} → {comp_check.index.max().date()}")
    print(f"  focal_wines_vwap_monthly:   {len(focal_check)} rows, "
          f"{focal_check['lwin7'].nunique() if 'lwin7' in focal_check.columns else '?'} wines")
    print(f"  top30_wines_vwap_monthly:   {len(top30_check)} rows, "
          f"{top30_check['lwin7'].nunique() if 'lwin7' in top30_check.columns else '?'} wines")
    print(f"  bids_monthly:               {len(bids_check)} rows")

## Summary

| Output file | Description | Frequency | Key columns |
|-------------|-------------|-----------|-------------|
| `livex_indices_clean.parquet` | Liv-ex index history (all numeric index columns) | Monthly (ME) | All index columns from CSV |
| `comparison_assets_monthly.parquet` | S&P 500, FTSE 100, Gold + Liv-ex aligned | Monthly (ME) | `sp500`, `ftse100`, `gold` + Liv-ex cols |
| `focal_wines_vwap_monthly.parquet` | VWAP for Salon, Dom Pérignon, Lafite (750ml) | Monthly | `month`, `lwin7`, `wine_name`, `vwap_gbp`, `trade_count` |
| `top30_wines_vwap_monthly.parquet` | Monthly VWAP for top 30 most-traded LWIN7s | Monthly | `month`, `lwin7`, `vwap_gbp`, `trade_count` |
| `bids_monthly.parquet` | Aggregated monthly bid prices (market-wide) | Monthly | `month`, `median_bid_gbp`, `bid_count` |

**MotherDuck tables schema documented in Section 2:**
- `winefi.ml.ml_unified_trades_tbvm`
- `winefi.mrt.mrt_unified_bids`

**All prices are in GBP.** Data filtered to 2000-01-01 onwards.  
Downstream notebooks read parquets from `../data/` (one level up from this notebook).